# Historical data preparation

This notebook preserves the code and available outputs from the original conversion, duplicate screening, split, and augmentation steps. It is an archive, not a prerequisite for each experiment run. The final counting cell was not executed in the historical record. For the fully executed experiment, see [CS156_Pipeline_Assignment_1.ipynb](CS156_Pipeline_Assignment_1.ipynb).

Private photos and filename mappings are excluded from the repository. Before rerunning, set the three `SOURCE` paths to your authorized local data and check every `OUTPUT` destination. Outputs already present on disk are protected by existence checks. Local directory prefixes in the source and logs have been replaced with portable placeholders for publication; historical result values are unchanged.


In [10]:
%pip install numpy pandas pillow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 83.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
from PIL import Image
import rawpy
import shutil

# 1. Define the source and output directories
SOURCE = Path("../data/assignment1")
OUTPUT = SOURCE.parent / (SOURCE.name + "_JPEG")

# 2. Check that the source directory exists
if not SOURCE.is_dir():
    raise FileNotFoundError(f"Source folder not found: {SOURCE}")

# Prevent overwriting an existing output directory
if OUTPUT.exists():
    raise FileExistsError(f"Output folder already exists: {OUTPUT}")

# 3. Validate the class directories
for label in ["positive", "negative"]:
    if not (SOURCE / label).is_dir():
        raise FileNotFoundError(f"Missing folder: {SOURCE / label}")

# 4. Process positive and negative images
for label in ["positive", "negative"]:

    input_folder = SOURCE / label
    output_folder = OUTPUT / label

    # Create the output directory
    output_folder.mkdir(parents=True)

    # Collect supported image files
    files = [
        f for f in input_folder.iterdir()
        if f.is_file() and f.suffix.lower() in
        [".arw", ".jpg", ".jpeg", ".png"]
    ]

    # Prioritize existing JPEG files to avoid unnecessary conversion
    priority = {
        ".jpg": 0,
        ".jpeg": 0,
        ".png": 1,
        ".arw": 2
    }

    files.sort(key=lambda f: (
        priority[f.suffix.lower()],
        f.name.lower()
    ))

    processed = set()
    count = 0

    for file in files:

        name = file.stem.lower()

        # Skip files with the same base filename
        if name in processed:
            print(f"Skipped duplicate name: {file.name}")
            continue

        output_file = output_folder / (file.stem + ".jpg")

        try:
            if file.suffix.lower() == ".arw":

                # Convert Sony RAW images to RGB
                with rawpy.imread(str(file)) as raw:
                    rgb = raw.postprocess(
                        use_camera_wb=True,
                        no_auto_bright=True,
                        output_color=rawpy.ColorSpace.sRGB,
                        output_bps=8
                    )

                # Save as high-quality JPEG
                Image.fromarray(rgb).save(
                    output_file,
                    "JPEG",
                    quality=95,
                    subsampling=0
                )

            elif file.suffix.lower() in [".jpg", ".jpeg"]:

                # Copy existing JPEG files without recompression
                shutil.copy2(file, output_file)

            elif file.suffix.lower() == ".png":

                # Convert PNG images to RGB JPEG
                with Image.open(file) as img:
                    img.convert("RGB").save(
                        output_file,
                        "JPEG",
                        quality=95,
                        subsampling=0
                    )

            processed.add(name)
            count += 1

        except Exception as e:
            print(f"Failed: {file.name} - {e}")

    # Display the number of processed images
    print(f"{label}: {count} images processed")

print("Conversion completed!")

positive: 62 images processed
negative: 45 images processed
Conversion completed!


In [7]:
from pathlib import Path
from PIL import Image
import imagehash
import shutil
import csv

# Define input and output directories
SOURCE = Path("../data/assignment1_JPEG")
OUTPUT = SOURCE.parent / "Tibet_Photos_Final"

# Check input directories before processing
if not SOURCE.is_dir():
    raise FileNotFoundError(SOURCE)

for label in ["positive", "negative"]:
    if not (SOURCE / label).is_dir():
        raise FileNotFoundError(SOURCE / label)

# Prevent overwriting existing files
if OUTPUT.exists():
    raise FileExistsError(f"Output already exists: {OUTPUT}")

# Collect image information
records = []

for label in ["positive", "negative"]:

    folder = SOURCE / label

    files = sorted(
        [f for f in folder.iterdir()
         if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg"]],
        key=lambda f: f.name.lower()
    )

    for file in files:

        # Calculate perceptual hash for duplicate detection
        with Image.open(file) as img:
            img.verify()

        with Image.open(file) as img:
            img.draft("RGB", (512, 512))
            img_hash = imagehash.phash(img)

        records.append({
            "path": file,
            "original_name": file.name,
            "label": label,
            "hash": img_hash
        })

# Identify potentially duplicate images
duplicates = []

for i in range(len(records)):
    for j in range(i + 1, len(records)):

        distance = records[i]["hash"] - records[j]["hash"]

        # A smaller hash distance indicates greater visual similarity
        if distance <= 4:
            duplicates.append([
                records[i]["label"],
                records[i]["original_name"],
                records[j]["label"],
                records[j]["original_name"],
                distance
            ])

print(f"Total images: {len(records)}")
print(f"Potential duplicate pairs: {len(duplicates)}")

for pair in duplicates:
    print("Possible duplicate:", pair)

# Create the final dataset directories
OUTPUT.mkdir()

for label in ["positive", "negative"]:
    (OUTPUT / label).mkdir()

# Rename and copy images
metadata = []

for label in ["positive", "negative"]:

    class_records = [
        r for r in records if r["label"] == label
    ]

    for index, record in enumerate(class_records, start=1):

        new_name = f"{label}_{index:03d}.jpg"
        destination = OUTPUT / label / new_name

        # Copy without modifying the original image
        shutil.copy2(record["path"], destination)

        metadata.append([
            record["original_name"],
            new_name,
            label
        ])

    print(f"{label}: {len(class_records)} images")

# Save filename mapping
with open(OUTPUT / "metadata.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["original_name", "new_name", "label"])
    writer.writerows(metadata)

# Save potential duplicate pairs
with open(OUTPUT / "duplicate_report.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "label_1", "image_1",
        "label_2", "image_2",
        "hash_distance"
    ])
    writer.writerows(duplicates)

print("\nDataset organization completed!")

Total images: 107
Potential duplicate pairs: 0
positive: 62 images
negative: 45 images

Dataset organization completed!


In [11]:
from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd
import shutil

# Define dataset paths
SOURCE = Path("../data/Tibet_Photos_Final")
OUTPUT = SOURCE.parent / "dataset"

# Set experiment parameters
TEST_SIZE = 0.20
SEED = 156

# Check source directory
if not SOURCE.is_dir():
    raise FileNotFoundError(SOURCE)

# Prevent overwriting existing data
if OUTPUT.exists():
    raise FileExistsError(f"Output already exists: {OUTPUT}")

# Collect image paths and labels
records = []

for label in ["positive", "negative"]:

    folder = SOURCE / label

    if not folder.is_dir():
        raise FileNotFoundError(folder)

    for image_path in sorted(folder.iterdir()):

        if image_path.is_file() and image_path.suffix.lower() in [".jpg", ".jpeg"]:

            records.append({
                "filename": image_path.name,
                "label": label,
                "path": str(image_path)
            })

df = pd.DataFrame(records)

# Split data while preserving class proportions
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df["label"],
    random_state=SEED
)

# Create output directories
for split in ["train", "test", "train_augmented"]:

    for label in ["positive", "negative"]:

        (OUTPUT / split / label).mkdir(
            parents=True,
            exist_ok=False
        )

# Copy images into train and test folders
for split_name, split_df in [
    ("train", train_df),
    ("test", test_df)
]:

    for _, row in split_df.iterrows():

        destination = OUTPUT / split_name / row["label"] / row["filename"]

        shutil.copy2(row["path"], destination)

# Save split metadata
train_df = train_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
test_df["split"] = "test"

split_metadata = pd.concat([train_df, test_df])

split_metadata.to_csv(
    OUTPUT / "split_metadata.csv",
    index=False
)

# Display split results
print("Training Set:")
print(train_df["label"].value_counts())

print("\nTest Set:")
print(test_df["label"].value_counts())

print("\nTrain/Test Split Completed!")

Training Set:
label
positive    49
negative    36
Name: count, dtype: int64

Test Set:
label
positive    13
negative     9
Name: count, dtype: int64

Train/Test Split Completed!


In [12]:
from PIL import Image, ImageEnhance, ImageOps
import numpy as np
import random
import shutil

# Set augmentation parameters
AUGMENTATIONS_PER_IMAGE = 9

random.seed(156)
rng = np.random.default_rng(156)


def add_gaussian_noise(image):

    # Convert the image to a writable NumPy array
    pixels = np.array(image)

    # Add noise in chunks to reduce memory usage
    for start in range(0, pixels.shape[0], 256):

        end = min(start + 256, pixels.shape[0])

        chunk = pixels[start:end].astype(np.float32)

        noise = rng.normal(
            0,
            0.01 * 255,
            chunk.shape
        )

        pixels[start:end] = np.clip(
            chunk + noise, 0, 255
        ).astype(np.uint8)

    return Image.fromarray(pixels)


def augment_image(image):

    img = image.copy()

    # Random brightness adjustment
    brightness = random.uniform(0.85, 1.15)
    img = ImageEnhance.Brightness(img).enhance(brightness)

    # Random contrast adjustment
    contrast = random.uniform(0.85, 1.15)
    img = ImageEnhance.Contrast(img).enhance(contrast)

    # Random horizontal flip
    if random.random() < 0.5:
        img = ImageOps.mirror(img)

    # Small random rotation
    angle = random.uniform(-5, 5)

    img = img.rotate(
        angle,
        resample=Image.Resampling.BICUBIC,
        expand=False,
        fillcolor=(127, 127, 127)
    )

    # Add mild Gaussian noise
    if random.random() < 0.5:
        img = add_gaussian_noise(img)

    return img


# Generate augmented training images
for label in ["positive", "negative"]:

    input_folder = OUTPUT / "train" / label
    output_folder = OUTPUT / "train_augmented" / label

    images = sorted(input_folder.glob("*.jpg"))

    for image_path in images:

        # Copy the original training image
        shutil.copy2(
            image_path,
            output_folder / image_path.name
        )

        # Load the image and correct EXIF orientation
        with Image.open(image_path) as image:

            original = ImageOps.exif_transpose(image).convert("RGB")

            # Generate nine augmented images
            for i in range(1, AUGMENTATIONS_PER_IMAGE + 1):

                augmented = augment_image(original)

                new_name = f"{image_path.stem}_aug_{i:02d}.jpg"

                augmented.save(
                    output_folder / new_name,
                    "JPEG",
                    quality=95,
                    subsampling=0
                )

                augmented.close()

        print(f"Processed: {image_path.name}")

    print(f"\n{label}: {len(images) * 10} images")

print("\n10x Data Augmentation Completed!")

Processed: positive_001.jpg
Processed: positive_002.jpg
Processed: positive_003.jpg
Processed: positive_004.jpg
Processed: positive_005.jpg
Processed: positive_006.jpg
Processed: positive_007.jpg
Processed: positive_008.jpg
Processed: positive_009.jpg
Processed: positive_010.jpg
Processed: positive_011.jpg
Processed: positive_012.jpg
Processed: positive_013.jpg
Processed: positive_014.jpg
Processed: positive_015.jpg
Processed: positive_017.jpg
Processed: positive_018.jpg
Processed: positive_019.jpg
Processed: positive_022.jpg
Processed: positive_023.jpg
Processed: positive_024.jpg
Processed: positive_026.jpg
Processed: positive_030.jpg
Processed: positive_031.jpg
Processed: positive_032.jpg
Processed: positive_034.jpg
Processed: positive_035.jpg
Processed: positive_036.jpg
Processed: positive_037.jpg
Processed: positive_039.jpg
Processed: positive_040.jpg
Processed: positive_042.jpg
Processed: positive_044.jpg
Processed: positive_045.jpg
Processed: positive_047.jpg
Processed: positive_

In [ ]:
# Count images in each dataset
for split in ["train", "test", "train_augmented"]:

    print(f"\n{split.upper()}")

    total = 0

    for label in ["positive", "negative"]:

        folder = OUTPUT / split / label

        count = len(list(folder.glob("*.jpg")))

        print(f"{label}: {count}")

        total += count

    print(f"Total: {total}")